In [ ]:
# default code from transformers website
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")
messages = [
    {"role": "user", "content": "Who are you?"},
]
inputs = tokenizer.apply_chat_template(
	messages,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=40)
print(tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:]))

In [ ]:
# take a look at this model's layers
print("QWEN model\n")
print(model)


print("\nUp proj")
proj = model.model.layers[0].mlp.up_proj
print(proj.weight.shape, proj.weight.dtype)

In [ ]:
import matplotlib.pyplot as plt
import torch

# first layer MLP layer up projection
W = model.model.layers[0].mlp.up_proj.weight.detach().float()
print(f"shape={tuple(W.shape)}, absmax={W.abs().max():.4f}, mean|w|={W.abs().mean():.4f}")
# 3072 rows - one for each output neuron
# 1024 cols - one for each input feature

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].hist(W.flatten().numpy(), bins = 200); ax[0].set_yscale("log")
ax[0].set_title("weight distribution")
ax[1].plot(W.abs().max(dim=1).values.numpy())
ax[1].set_title('per-row absmax(outlier hunting)')
plt.show()

## Demo of quantizing to e2m1

In [ ]:
def e2m1_values():
    """SEEM layout with exponent bias = 1

    S     E    E      M
    |     |           |
    sign exponent mantissa

    """

    vals = []
    for code in range(16):
        sign = (code >> 3) & 1 # shifts right and then mask the last bit
        exp = (code >> 1) & 3 # shifts right 1 and mask to get lowest 2 bits
        mant = code & 1 # masks lowest bit of original

        if exp == 0: # subnormal: value = 0.5 * mant
            v = 0.5 * mant
        
        else: # normal: 2^(exp - 1) * (1 + 0.5 * mant)
            v = (2 ** (exp - 1)) * (1 + 0.5 * mant)
        
        vals.append(-v if sign else v)
    
    return vals

for code, v in enumerate(e2m1_values()):
    print(f"{code:04b} {v:+.2f}")

In [ ]:
import numpy as np

fp4_levels = np.array(sorted(set(e2m1_values())))  # 15 unique values
FP4_MAX = 6.0

def quantize_block_absmax(x, levels=fp4_levels, fp4_max=FP4_MAX):
    scale = np.abs(x).max() / fp4_max
    if scale == 0: scale = 1.0
    scaled = x / scale
    idx = np.abs(scaled[:, None] - levels[None, :]).argmin(axis=1)
    return levels[idx], scale

block = W[0, :32].numpy()
q, s = quantize_block_absmax(block)
dq = q * s

print(f"original : {block[:8].round(3)}")
print(f"dequant  : {dq[:8].round(3)}")
print(f"MSE={((block-dq)**2).mean():.6f}  relMSE={((block-dq)**2).mean()/(block**2).mean():.4f}")

In [ ]:
def absmax_qdq(W_np, block_size, levels=fp4_levels, fp4_max=FP4_MAX):
    rows, K = W_np.shape
    nb = K // block_size
    Wb = W_np[:, :nb*block_size].reshape(rows, nb, block_size)
    s  = np.abs(Wb).max(-1, keepdims=True) / fp4_max
    s  = np.where(s == 0, 1.0, s)
    idx = np.abs((Wb / s)[..., None] - levels).argmin(-1)
    return (levels[idx] * s).reshape(rows, nb*block_size)

W_np = W.numpy()
for bs in [16, 32, 64, 128, 256]:
    dq = absmax_qdq(W_np, bs)
    orig = W_np[:, :dq.shape[1]]
    rel = ((orig - dq)**2).mean() / (orig**2).mean()
    print(f"bs={bs:4d}  relMSE={rel:.4f}  scale overhead={32/bs:.2f} bits/wt (fp32 scales)")

In [ ]:
import torch.nn as nn
import copy

class FakeQuantLinear(nn.Module):
    def __init__(self, linear, block_size=32):
        super().__init__()
        W = linear.weight.detach().float().cpu().numpy()
        # pad K to multiple of block_size, then trim back
        pad = (-W.shape[1]) % block_size
        Wp  = np.pad(W, ((0,0),(0,pad)))
        dq  = absmax_qdq(Wp, block_size)[:, :W.shape[1]]
        self.weight = nn.Parameter(
            torch.from_numpy(dq).to(linear.weight.dtype).to(linear.weight.device),
            requires_grad=False,
        )
        self.bias = linear.bias

    def forward(self, x):
        return nn.functional.linear(x, self.weight, self.bias)

# snapshot original generation for comparison
out_bf16 = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print("BF16    :", tokenizer.decode(out_bf16[0][inputs["input_ids"].shape[-1]:]))

# swap just ONEE layer's up_proj
orig = model.model.layers[0].mlp.up_proj
model.model.layers[0].mlp.up_proj = FakeQuantLinear(orig, block_size=32)

out_fp4 = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print("FP4 L0  :", tokenizer.decode(out_fp4[0][inputs["input_ids"].shape[-1]:]))

# restore
model.model.layers[0].mlp.up_proj = orig

In [ ]:
def swap_all(model, block_size, which):
    """which is a list like ['up_proj', 'gate_proj', 'down_proj', 'q_proj', ...]"""
    originals = {}
    for i, layer in enumerate(model.model.layers):
        for name in which:
            for parent_name in ['mlp', 'self_attn']:
                parent = getattr(layer, parent_name, None)
                if parent is not None and hasattr(parent, name):
                    mod = getattr(parent, name)
                    originals[(i, parent_name, name)] = mod
                    setattr(parent, name, FakeQuantLinear(mod, block_size))
    return originals

def restore(model, originals):
    for (i, p, n), mod in originals.items():
        setattr(getattr(model.model.layers[i], p), n, mod)

# try: all MLPs only
orig = swap_all(model, 32, ['up_proj', 'gate_proj', 'down_proj'])
out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print("all MLP fp4 :", tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:]))
restore(model, orig)

# then: all attention only — often more sensitive
orig = swap_all(model, 32, ['q_proj', 'k_proj', 'v_proj', 'o_proj'])
out = model.generate(**inputs, max_new_tokens=40, do_sample=False)
print("all attn fp4:", tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:]))
restore(model, orig)